# Assignment 8 - Noisy channel spell correction

    This notebook reads the three supplied files, constructs one-edit candidates from the unigram vocabulary, scores each character change using `count_1edit.txt`, and evaluates top-1 accuracy against `spell-errors.txt`. The noisy-channel decision is `argmax_c P(error|c) P(c)`.

## 1. Load and inspect the supplied data

    Paths are resolved relative to this notebook, so the notebook can be run from any working directory. The training examples are formatted as `correct: error1, error2*count, ...`; an omitted multiplier means one occurrence.

In [3]:
from pathlib import Path
from collections import Counter, defaultdict
import re
import math

DATA_DIR = Path("Spell-Correction")
if not DATA_DIR.exists():
    DATA_DIR = Path("Assignment8") / "Spell-Correction"
if not DATA_DIR.exists():
    raise FileNotFoundError("Could not find the Assignment8/Spell-Correction data folder")

def parse_counted_list(path):
    table = defaultdict(Counter)
    for line_no, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip() or ":" not in line:
            continue
        correct, errors = line.split(":", 1)
        correct = correct.strip().lower()
        for item in errors.split(","):
            item = item.strip().lower()
            if not item:
                continue
            match = re.fullmatch(r"(.+?)(?:\*(\d+))?", item)
            if match:
                error, count = match.group(1), int(match.group(2) or 1)
                table[error][correct] += count
    return table

gold_error_to_correct = parse_counted_list(DATA_DIR / "spell-errors.txt")

unigram = {}
for line in (DATA_DIR / "count_unigram.txt").read_text(encoding="utf-8").splitlines():
    fields = line.split()
    if len(fields) >= 2 and fields[-1].isdigit():
        unigram[fields[0].lower()] = int(fields[-1])

confusion = Counter()
for line in (DATA_DIR / "count_1edit.txt").read_text(encoding="latin-1").splitlines():
    fields = line.split()
    if len(fields) >= 2 and "|" in fields[0] and fields[-1].isdigit():
        intended, observed = fields[0].split("|", 1)
        confusion[(intended.lower(), observed.lower())] += int(fields[-1])

{"gold words": len(gold_error_to_correct),
 "gold error-correction pairs": sum(map(len, gold_error_to_correct.values())),
 "character confusion pairs": len(confusion),
 "unigram vocabulary": len(unigram),
 "sample gold entry": next(iter(gold_error_to_correct.items())),
 "sample confusion pairs": list(confusion.items())[:5],
 "sample unigram entries": list(unigram.items())[:5]}

{'gold words': 36978, 'gold error-correction pairs': 39590, 'character confusion pairs': 1511, 'unigram vocabulary': 333333, 'sample gold entry': ('rainning', Counter({'raining': 1})), 'sample confusion pairs': [(('e', 'i'), 917), (('a', 'e'), 865), (('i', 'e'), 773), (('e', 'a'), 749), (('a', 'i'), 560)], 'sample unigram entries': [('the', 23135851162), ('of', 13151942776), ('and', 12997637966), ('to', 12136980858), ('a', 9081174698)]}

**Interpretation.** Each parsed gold entry maps one misspelling to the intended word or words. `*k` is retained as a count, not as part of the token. The one-edit file contains character-fragment confusion frequencies, while the unigram file supplies the vocabulary and word frequency.

## 2. Noisy channel scoring

    The one-edit file stores `intended-fragment|observed-fragment` counts. For each typo, generate all one-character insertions, deletions, substitutions, and adjacent transpositions, retaining candidates in the unigram vocabulary. Estimate `P(error|correct)` from the relevant fragment count divided by all confusions with the same intended fragment; use a small floor for unseen changes. Estimate `P(correct)` from unigram frequencies. Scores are computed in log space.

In [6]:
confusion_totals = Counter()
for (intended, observed), count in confusion.items():
    confusion_totals[intended] += count
unigram_total = sum(unigram.values())

def one_edit_candidates(error):
    edits = set()
    letters = "abcdefghijklmnopqrstuvwxyz"
    for i in range(len(error)):
        edits.add(error[:i] + error[i+1:])
        for ch in letters:
            edits.add(error[:i] + ch + error[i+1:])
    for i in range(len(error) + 1):
        for ch in letters:
            edits.add(error[:i] + ch + error[i:])
    for i in range(len(error) - 1):
        edits.add(error[:i] + error[i+1] + error[i] + error[i+2:])
    return {w for w in edits if w in unigram and w != error}

def edit_confusion(correct, error):
    choices = []
    for start in range(min(len(correct), len(error)) + 1):
        if correct[:start] != error[:start]:
            continue
        for c_end in range(start, min(len(correct), start + 2) + 1):
            for e_end in range(start, min(len(error), start + 2) + 1):
                if correct[c_end:] == error[e_end:] and (c_end-start, e_end-start) != (0, 0):
                    key = (correct[start:c_end], error[start:e_end])
                    choices.append((confusion.get(key, 0), key))
    return max(choices, default=(0, ("", "")), key=lambda item: item[0])[1]

def score_candidates(error):
    candidates = one_edit_candidates(error.lower())
    scored = []
    for correct in candidates:
        prior_count = unigram.get(correct, 0)
        key = edit_confusion(correct, error.lower())
        channel_count = confusion.get(key, 0)
        denom = confusion_totals[key[0]]
        channel_prob = channel_count / denom if channel_count and denom else 1 / max(sum(confusion.values()), 1)
        log_score = math.log(channel_prob) + math.log(prior_count / unigram_total)
        scored.append((correct, log_score, channel_count, prior_count, key))
    return sorted(scored, key=lambda row: (-row[1], row[0]))

def correct_word(error):
    ranked = score_candidates(error)
    return ranked[0][0] if ranked else error.lower()

demo_errors = ["rainning", "raning", "yello", "forer"]
{error: {"ranking": score_candidates(error)[:5], "prediction": correct_word(error)}
 for error in demo_errors}

{'rainning': {'ranking': [('raining', -15.807294463415152, 146, 1392293, ('i', 'in')), ('trainning', -27.18796042789183, 0, 35444, ('t', ''))], 'prediction': 'raining'}, 'raning': {'ranking': [('ranking', -10.889707349913495, 9, 14624061, ('nk', 'n')), ('ranging', -11.21234089196861, 35, 10213057, ('ng', 'n')), ('rating', -12.740866903104383, 12, 207858692, ('t', 'n')), ('raining', -13.160931058493782, 139, 1392293, ('ai', 'a')), ('ranting', -13.66547376381649, 71, 683328, ('nt', 'n'))], 'prediction': 'ranking'}, 'yello': {'ranking': [('yellow', -8.988901615225137, 51, 82024459, ('ow', 'o')), ('yell', -15.567368998638784, 42, 1654034, ('l', 'lo')), ('hello', -15.752966542568156, 1, 32960381, ('h', 'y')), ('yelloe', -16.734420267355823, 25, 36835, ('oe', 'o')), ('yelloa', -16.90418745642389, 53, 28313, ('oa', 'o'))], 'prediction': 'yellow'}, 'forer': {'ranking': [('former', -9.436301380425258, 3, 46917563, ('rm', 'r')), ('fore', -15.387109403519053, 211, 2378954, ('e', 'er')), ('forger'

**Interpretation.** Rankings list candidate, log score, confusion count, unigram frequency, and the character fragments used. A larger score means the candidate better explains the typo while also being more common in the unigram corpus.

## 3. Evaluate correction accuracy

    Evaluation uses every error-correction pair listed in `spell-errors.txt`. For each misspelling, the first-ranked correction is compared with each gold intended word. If a misspelling has multiple acceptable gold corrections, it counts as correct if the prediction matches any one of them. Accuracy is the number of correct predictions divided by the number of evaluated pairs. Coverage is the fraction of distinct gold misspellings with a candidate in the unigram vocabulary.

In [9]:
evaluated = []
for error, gold_counts in gold_error_to_correct.items():
    prediction = correct_word(error)
    covered = bool(one_edit_candidates(error))
    for gold, multiplicity in gold_counts.items():
        for _ in range(multiplicity):
            evaluated.append((error, gold, prediction, covered))

correct_n = sum(pred == gold for _, gold, pred, _ in evaluated)
total_n = len(evaluated)
accuracy = correct_n / total_n if total_n else float("nan")
coverage = sum(bool(one_edit_candidates(e)) for e in gold_error_to_correct) / len(gold_error_to_correct)
errors = [(e, gold, pred) for e, gold, pred, _ in evaluated if pred != gold]

{"evaluated pairs": total_n, "correct top-1 predictions": correct_n,
 "accuracy": round(accuracy, 4), "coverage of distinct misspellings": round(coverage, 4),
 "incorrect examples (first 20)": errors[:20]}

{'evaluated pairs': 41718, 'correct top-1 predictions': 12511, 'accuracy': 0.2999, 'coverage of distinct misspellings': 0.6639, 'incorrect examples (first 20)': [('raning', 'raining', 'ranking'), ('disparingly', 'disparagingly', 'disparingly'), ('forer', 'four', 'former'), ('fuore', 'four', 'furore'), ('fore', 'four', 'for'), ('fore', 'four', 'for'), ('fore', 'four', 'for'), ('fore', 'four', 'for'), ('fore', 'four', 'for'), ('fore', 'fight', 'for'), ('fore', 'form', 'for'), ('for', 'four', 'form'), ('for', 'four', 'form'), ('for', 'four', 'form'), ('for', 'four', 'form'), ('for', 'flying', 'form'), ('for', 'before', 'form'), ('for', 'of', 'form'), ('for', 'far', 'form'), ('for', 'from', 'form')]}

**How to interpret the result.** Accuracy is a top-1 exact-word match on the supplied benchmark, and coverage distinguishes model errors from missing candidate entries. The final cell shows representative mistakes for inspection. This is a closed candidate-set evaluation: the model cannot recover a correct word absent from the unigram vocabulary or outside the generated one-edit neighborhood. The score is specific to these files and this channel normalization; it should not be interpreted as accuracy on arbitrary real-world text. Character confusion likelihoods and unigram frequencies provide the channel and word prior, respectively.